# Daily Practice — 2026-09-21 — ML Testing & MLOps: Building a Drift Monitoring Test Suite

**Dataset:** [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) — 398 real U.S. EPA fuel-economy
records for cars sold 1970–1982 (mpg, cylinders, displacement, horsepower, weight, acceleration,
model year, origin), loaded from the `mwaskom/seaborn-data` mirror of the classic UCI dataset.

## Problem statement

Imagine a regression model was trained to predict `mpg` from a car's specs, using only cars sold in
**1970–1976**. It's now "in production," scoring cars from **1977–1982**. In between those two
periods, the 1975 CAFE fuel-economy standards phased in and the 1979 oil crisis hit — automakers
responded by making cars lighter, smaller-engined, and more efficient. That's a real, well-documented
covariate shift, not a synthetic one.

As the QA/MLOps engineer on this project, your job is to build the automated checks that *would have
caught this drift* before it silently degraded the model — the kind of checks that belong in a
monitoring pipeline that runs on every new batch of scored data, long before anyone manually notices
the model looks "off."

## What you should produce

1. A reference-period (1970–76) train/test split, a fitted `RandomForestRegressor` baseline, and its
   held-out MAE.
2. A `compute_psi` function (Population Stability Index) and a `feature_drift_report` that applies it
   to every input feature, comparing the production period (1977–82) against the reference training
   distribution, and flags any feature with PSI ≥ 0.2 ("moderate-to-significant shift" by the usual
   industry convention: <0.1 no meaningful shift, 0.1–0.25 moderate, ≥0.25 significant).
3. A `performance_degradation_test` that compares the model's MAE on the production period against its
   reference held-out MAE, with a configurable relative-increase gate (e.g. 15%), returning a pass/fail
   verdict.
4. A `prediction_drift_test` that checks whether the *distribution of the model's own predictions* has
   shifted (PSI computed on predicted values, reference vs. production) — a check that still works even
   when you don't have production ground-truth labels yet, which is the realistic monitoring situation.
5. A short written verdict (3–5 sentences): which check would fire first in a real deployment where
   labels typically arrive late or never, and whether relying on the model's own historical held-out
   accuracy as the *only* CI/release gate would have caught this failure before it hit production.

Try it yourself in the starter cells below before you scroll down to the solution.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42

## Load the data and define the reference / production split

We split on `model_year` itself — this is a genuine temporal split, not injected noise. Six rows have
a missing `horsepower` (recorded as `?` in the original UCI data) and are dropped. `origin` and `name`
are excluded from the model's numeric features; `model_year` is excluded too, since it's the variable
we're splitting on and including it would trivially "predict" the split.

In [ ]:
MPG_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

df = pd.read_csv(MPG_URL).dropna().reset_index(drop=True)

FEATURES = ["cylinders", "displacement", "horsepower", "weight", "acceleration"]
TARGET = "mpg"

# Reference period: pre-CAFE-standards cars (1970-76). Production period: post-oil-crisis cars (1977-82).
reference_df = df[df["model_year"] < 77].reset_index(drop=True)
production_df = df[df["model_year"] >= 77].reset_index(drop=True)

X_ref, y_ref = reference_df[FEATURES], reference_df[TARGET]
X_prod, y_prod = production_df[FEATURES], production_df[TARGET]

print(f"reference: {X_ref.shape}, production: {X_prod.shape}")
reference_df[FEATURES + [TARGET]].describe()

## Your task

Fill in the TODOs below. Signatures are given — don't change them, the solution cells assume these
names.

In [ ]:
# TODO: split X_ref, y_ref into X_ref_train, X_ref_test, y_ref_train, y_ref_test
# - 70/30 split
# - random_state=RANDOM_STATE
# (no need to stratify — this is a regression target)

X_ref_train, X_ref_test, y_ref_train, y_ref_test = None, None, None, None

In [ ]:
def compute_psi(reference, production, buckets=10):
    """TODO: compute the Population Stability Index between two 1-D numeric arrays.

    Steps:
    1. Build `buckets` bin edges from the quantiles of `reference` only (np.percentile at
       np.linspace(0, 100, buckets + 1)), then de-duplicate them (np.unique) since ties can
       collapse quantile edges. Replace the first edge with -inf and the last with +inf so
       every production value falls into some bin.
    2. Histogram both `reference` and `production` into those same edges (np.histogram).
    3. Convert counts to proportions of each array's own length. Clip proportions away from
       zero (e.g. np.clip(..., 1e-4, None)) so the log term below never blows up.
    4. Return sum((prod_pct - ref_pct) * log(prod_pct / ref_pct)) as a float.
    """
    raise NotImplementedError

In [ ]:
def feature_drift_report(X_ref, X_prod, psi_threshold=0.2):
    """TODO: for every column in X_ref, compute compute_psi(X_ref[col], X_prod[col]) and
    return a pandas DataFrame with columns ['feature', 'psi', 'drifted'] (drifted = psi >=
    psi_threshold), sorted by psi descending.
    """
    raise NotImplementedError

In [ ]:
# TODO: call feature_drift_report(X_ref_train, X_prod) and display the result.

drift_report = None

In [ ]:
def fit_reference_model(X_ref_train, y_ref_train):
    """TODO: fit and return a sklearn RandomForestRegressor on X_ref_train/y_ref_train.
    Use n_estimators=200, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1.
    """
    raise NotImplementedError

In [ ]:
def performance_degradation_test(ref_mae, prod_mae, rel_threshold=0.15):
    """TODO: given the reference held-out MAE and the production MAE, compute the relative
    increase pct = (prod_mae - ref_mae) / ref_mae and return a dict with keys 'ref_mae',
    'prod_mae', 'pct_increase', 'passed' (passed = pct <= rel_threshold).
    """
    raise NotImplementedError

In [ ]:
# TODO:
# 1. Fit the reference model with fit_reference_model.
# 2. Compute ref_mae with mean_absolute_error on (y_ref_test, model.predict(X_ref_test)).
# 3. Compute prod_mae with mean_absolute_error on (y_prod, model.predict(X_prod)).
# 4. Call performance_degradation_test(ref_mae, prod_mae) and print the result.

reference_model = None
ref_mae = None
prod_mae = None
performance_result = None

In [ ]:
def prediction_drift_test(model, X_ref_test, X_prod, psi_threshold=0.2):
    """TODO: get model.predict(X_ref_test) and model.predict(X_prod), compute their PSI with
    compute_psi, and return a dict with keys 'psi' and 'drifted' (psi >= psi_threshold).

    This check needs no production labels at all -- only the model's own outputs on old vs.
    new inputs -- which is what makes it usable the moment production data starts flowing,
    well before ground truth is available.
    """
    raise NotImplementedError

In [ ]:
# TODO: call prediction_drift_test(reference_model, X_ref_test, X_prod) and print the result.

prediction_drift_result = None

### Write-up

TODO: in 3–5 sentences, say which of the three checks (feature drift, performance degradation,
prediction drift) would fire *first* in a real deployment where production labels typically arrive
late or not at all, and whether relying on the model's own historical held-out MAE as the only
CI/release gate would have caught this failure before it reached production.

*(your answer here)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. Split the reference period
X_ref_train, X_ref_test, y_ref_train, y_ref_test = train_test_split(
    X_ref, y_ref, test_size=0.3, random_state=RANDOM_STATE
)

# 2. Population Stability Index
def compute_psi(reference, production, buckets=10):
    reference = np.asarray(reference, dtype=float)
    production = np.asarray(production, dtype=float)
    quantiles = np.linspace(0, 100, buckets + 1)
    breakpoints = np.unique(np.percentile(reference, quantiles))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    ref_counts, _ = np.histogram(reference, bins=breakpoints)
    prod_counts, _ = np.histogram(production, bins=breakpoints)
    ref_pct = np.clip(ref_counts / len(reference), 1e-4, None)
    prod_pct = np.clip(prod_counts / len(production), 1e-4, None)
    return float(np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct)))

# 3. Feature-level drift report
def feature_drift_report(X_ref, X_prod, psi_threshold=0.2):
    rows = []
    for col in X_ref.columns:
        psi = compute_psi(X_ref[col], X_prod[col])
        rows.append({"feature": col, "psi": round(psi, 4), "drifted": psi >= psi_threshold})
    return pd.DataFrame(rows).sort_values("psi", ascending=False).reset_index(drop=True)

drift_report = feature_drift_report(X_ref_train, X_prod)
print(drift_report)

# 4. Reference model
def fit_reference_model(X_ref_train, y_ref_train):
    model = RandomForestRegressor(
        n_estimators=200, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_ref_train, y_ref_train)
    return model

reference_model = fit_reference_model(X_ref_train, y_ref_train)
ref_mae = mean_absolute_error(y_ref_test, reference_model.predict(X_ref_test))
prod_mae = mean_absolute_error(y_prod, reference_model.predict(X_prod))

# 5. Performance degradation gate
def performance_degradation_test(ref_mae, prod_mae, rel_threshold=0.15):
    pct = (prod_mae - ref_mae) / ref_mae
    return {
        "ref_mae": ref_mae,
        "prod_mae": prod_mae,
        "pct_increase": pct,
        "passed": pct <= rel_threshold,
    }

performance_result = performance_degradation_test(ref_mae, prod_mae)
print(performance_result)

# 6. Prediction drift (label-free)
def prediction_drift_test(model, X_ref_test, X_prod, psi_threshold=0.2):
    pred_ref = model.predict(X_ref_test)
    pred_prod = model.predict(X_prod)
    psi = compute_psi(pred_ref, pred_prod)
    return {"psi": psi, "drifted": psi >= psi_threshold}

prediction_drift_result = prediction_drift_test(reference_model, X_ref_test, X_prod)
print(prediction_drift_result)
```

**Typical output**

```
        feature     psi  drifted
0        weight  0.9978     True
1    horsepower  0.8555     True
2  displacement  0.7192     True
3  acceleration  0.4535     True
4     cylinders  0.2995     True

{'ref_mae': 1.68, 'prod_mae': 5.32, 'pct_increase': 2.17, 'passed': False}
{'psi': 0.415, 'drifted': True}
```

**Explanation**

Every single input feature comes back flagged, several with PSI well above the "significant shift"
line of 0.25 — `weight` and `horsepower` shift the hardest, which lines up exactly with what actually
happened to the U.S. car market between 1976 and 1982: the 1975 CAFE standards and the 1979 oil crisis
pushed automakers toward smaller engines and lighter bodies. The reference model's held-out MAE is
about 1.7 mpg, but its MAE on production data balloons to about 5.3 mpg — a 217% relative increase,
which blows straight through a 15% degradation gate. The prediction-drift check (PSI = 0.42 on the
model's own outputs, no labels required) also fires, and would have been available *immediately* when
production scoring started, unlike the performance-degradation check, which needs ground-truth `mpg`
values that in a real pipeline might not arrive for days or weeks after a car is scored.

That ordering is the point: in practice, feature drift and prediction drift are your early-warning
system, and performance degradation is the (delayed) confirmation. A CI gate that only checks
"held-out test MAE on the *original* 1970-76 test set is below some threshold" would never re-run
against new data at all — it would pass forever, at the same 1.7 MAE, no matter how badly the model
degrades in production, because nothing about that gate depends on what the production inputs actually
look like. Catching this kind of failure requires monitoring checks that run continuously against
*live* input and (eventually) output data, not a single held-out accuracy number frozen at training
time.

</details>